In [ ]:
from dotenv import load_dotenv

load_dotenv()

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🔍 语义搜索（Semantic Search）

**一句话定位**：把一份 PDF 切块、embedding、存进向量库，再用「语义相似度」而不是关键词去检索——这是 RAG 的第一半。



</div>

流程是：**加载 PDF（`PyPDFLoader`）→ 切块（`RecursiveCharacterTextSplitter`）→ 向量化（`OpenAIEmbeddings`）→ 存入向量库（`InMemoryVectorStore`）→ 相似度检索（`similarity_search`）**。

`chunk_overlap=200` 这种重叠切块是为了避免一个完整的语义单元被硬切断在两个 chunk 的边界上，导致检索时两边都拿不到完整信息。

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("resources/acmecorp-employee-handbook.pdf")

data = loader.load()

print(data)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)

print(len(all_splits))

📎 Embedding 模型可选项参考官方文档：[Text Embedding Models](https://docs.langchain.com/oss/python/integrations/text_embedding)——这里用的是 `text-embedding-3-large`，换成别的模型（甚至别的厂商）不影响下游 pipeline 的代码结构。

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [ ]:
ids = vector_store.add_documents(documents=all_splits)

In [ ]:
results = vector_store.similarity_search(
    "How many days of vacation does an employee get in their first year?"
)

print(results[0])

## 🤖 RAG Agent

**把检索包装成一个工具**：`search_handbook` 内部调用 `vector_store.similarity_search(query)`，agent 需要员工手册信息时就调用这个工具，而不是在 system prompt 里塞进整本手册。

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

这正好和 2.2_runtime_context 里讲的「懒加载、按需访问」原理相通——RAG 本质上就是把「外部知识」当成一种可按需查询的 context 来源，通过工具调用取回**相关片段**，而不是整份文档一次性塞进 prompt。

</div>

In [ ]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee handbook for information."
    )

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many days of vacation does an employee get in their first year?")]}
)

In [ ]:
from pprint import pprint

pprint(response)